# 3. Autoregressive Text Generation

The previous notebook ended with a next-token distribution. We now turn that single prediction into text generation by repeatedly selecting one token, appending it to the sequence and running the model again. We will first implement this loop explicitly, then compare it with `model.generate` before introducing stochastic decoding and the KV cache.

The core process is:

```text
prompt -> forward -> final logits -> select token -> append -> repeat
```

We keep the model, prompt and token budget fixed while changing one decoding decision at a time. This lets us attribute differences in the generated text to greedy decoding, temperature, top-k or caching rather than to an uncontrolled change in the experiment.

## 1. Reproduce the local model setup

This notebook starts from a fresh kernel and recreates every dependency explicitly. We load the tokenizer and the same immutable Qwen3 base checkpoint from the local cache, select MPS with BF16 and place the model in evaluation mode. No variable from the previous notebook is required.

Our prompt is an unfinished textual prefix rather than an instruction because this is a base model trained for continuation. `MAX_NEW_TOKENS` limits how many generation steps we allow; it is different from the model's context limit, which bounds the combined prompt and generated sequence. We use `model.config.max_position_embeddings` as the architectural limit and verify that the planned experiment fits inside it. The seed will matter once we sample, although greedy decoding itself contains no random choice.

In [1]:
import json
import platform
import sys
from datetime import datetime, timezone
from importlib.metadata import version

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
REVISION = "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"
PROMPT = "In a small village at the edge of the sea"
MAX_NEW_TOKENS = 32
SEED = 123

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    MODEL_DTYPE = torch.bfloat16
else:
    DEVICE = torch.device("cpu")
    MODEL_DTYPE = torch.float32

torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    local_files_only=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    local_files_only=True,
    dtype=MODEL_DTYPE,
).to(DEVICE)
model.eval()

prompt_inputs = tokenizer(
    PROMPT,
    add_special_tokens=False,
    return_tensors="pt",
).to(DEVICE)
prompt_length = prompt_inputs["input_ids"].shape[1]
context_limit = model.config.max_position_embeddings
assert prompt_length + MAX_NEW_TOKENS <= context_limit

setup = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "os": platform.platform(),
    "packages": {
        name: version(name)
        for name in ("torch", "transformers", "huggingface-hub", "safetensors")
    },
    "model_id": MODEL_ID,
    "revision": REVISION,
    "device": str(DEVICE),
    "runtime_dtype": str(MODEL_DTYPE),
    "seed": SEED,
    "prompt": PROMPT,
    "prompt_token_ids": prompt_inputs["input_ids"][0].tolist(),
    "prompt_tokens": tokenizer.convert_ids_to_tokens(
        prompt_inputs["input_ids"][0].tolist()
    ),
    "prompt_length": prompt_length,
    "max_new_tokens": MAX_NEW_TOKENS,
    "context_limit": context_limit,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.pad_token_id,
}
print(json.dumps(setup, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

{
  "started_utc": "2026-10-12T11:05:39.647087+00:00",
  "python": "3.12.13",
  "os": "macOS-26.6.2-arm64-arm-64bit",
  "packages": {
    "torch": "2.11.0",
    "transformers": "5.19.0",
    "huggingface-hub": "2.2.0",
    "safetensors": "0.8.0"
  },
  "model_id": "Qwen/Qwen3-0.6B-Base",
  "revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
  "device": "mps",
  "runtime_dtype": "torch.bfloat16",
  "seed": 123,
  "prompt": "In a small village at the edge of the sea",
  "prompt_token_ids": [
    641,
    264,
    2613,
    14126,
    518,
    279,
    6821,
    315,
    279,
    9396
  ],
  "prompt_tokens": [
    "In",
    "Ġa",
    "Ġsmall",
    "Ġvillage",
    "Ġat",
    "Ġthe",
    "Ġedge",
    "Ġof",
    "Ġthe",
    "Ġsea"
  ],
  "prompt_length": 10,
  "max_new_tokens": 32,
  "context_limit": 32768,
  "eos_token_id": 151643,
  "pad_token_id": 151643
}


## 2. Implement manual greedy decoding

We turn one next-token prediction into generation with an explicit loop. At each step, we run the complete sequence through the model, select the final-position logits, take their argmax and append the resulting `[B, 1]` token tensor to `input_ids`. The attention mask grows by one real position at the same time. Softmax is unnecessary because it preserves the ordering of the logits and therefore does not change the argmax.

This first implementation deliberately disables the KV cache. If the sequence grows from `T` to `T + 1`, the next forward recomputes all previous token positions as well as the new one. That is inefficient, but it exposes the autoregressive algorithm without hiding state inside a cache. We stop when the model emits EOS or when we exhaust the token budget, and we reject prompts whose planned generation would exceed the context limit.

The function accepts a new prompt without reloading the model, so after this cell we can test other textual prefixes by calling `generate_greedy` again.

In [2]:
def generate_greedy(prompt, max_new_tokens):
    encoded = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(DEVICE)
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    prompt_length = input_ids.shape[1]

    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be at least 1")
    if prompt_length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("Prompt and generation exceed the model context limit")

    steps = []
    stopped_on_eos = False

    for step in range(max_new_tokens):
        input_shape = tuple(input_ids.shape)
        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            )

        final_position_logits = outputs.logits[:, -1, :]
        next_token_id = torch.argmax(
            final_position_logits,
            dim=-1,
            keepdim=True,
        )
        token_id = int(next_token_id.item())
        if token_id >= len(tokenizer):
            raise RuntimeError(f"Model selected unmapped token ID {token_id}")

        input_ids = torch.cat((input_ids, next_token_id), dim=1)
        attention_mask = torch.cat(
            (attention_mask, torch.ones_like(next_token_id)),
            dim=1,
        )

        steps.append({
            "step": step + 1,
            "input_shape": input_shape,
            "next_token_shape": tuple(next_token_id.shape),
            "token_id": token_id,
            "token_piece": tokenizer.decode([token_id], skip_special_tokens=False),
            "new_sequence_length": input_ids.shape[1],
        })

        if token_id == tokenizer.eos_token_id:
            stopped_on_eos = True
            break

    generated_ids = input_ids[0, prompt_length:].cpu().tolist()
    return {
        "prompt": prompt,
        "prompt_length": prompt_length,
        "generated_ids": generated_ids,
        "generated_tokens": len(generated_ids),
        "stopped_on_eos": stopped_on_eos,
        "steps": steps,
        "text": tokenizer.decode(input_ids[0].cpu(), skip_special_tokens=True),
    }


manual_greedy = generate_greedy(PROMPT, MAX_NEW_TOKENS)

print(f"Prompt length: {manual_greedy['prompt_length']} tokens")
print(f"Generated:     {manual_greedy['generated_tokens']} tokens")
print(f"Stopped on EOS: {manual_greedy['stopped_on_eos']}")
print(f"\n{'Step':>4} {'Input [B,T]':>14} {'Next [B,1]':>12} {'ID':>8} {'New T':>6}  Piece")
for record in manual_greedy["steps"]:
    print(
        f"{record['step']:>4} {str(record['input_shape']):>14} "
        f"{str(record['next_token_shape']):>12} {record['token_id']:>8} "
        f"{record['new_sequence_length']:>6}  {record['token_piece']!r}"
    )
print(f"\nGenerated text:\n{manual_greedy['text']}")

Prompt length: 10 tokens
Generated:     32 tokens
Stopped on EOS: False

Step    Input [B,T]   Next [B,1]       ID  New T  Piece
   1        (1, 10)       (1, 1)       11     11  ','
   2        (1, 11)       (1, 1)     1052     12  ' there'
   3        (1, 12)       (1, 1)      374     13  ' is'
   4        (1, 13)       (1, 1)      264     14  ' a'
   5        (1, 14)       (1, 1)     2613     15  ' small'
   6        (1, 15)       (1, 1)    12922     16  ' island'
   7        (1, 16)       (1, 1)      448     17  ' with'
   8        (1, 17)       (1, 1)      264     18  ' a'
   9        (1, 18)       (1, 1)     7042     19  ' population'
  10        (1, 19)       (1, 1)      315     20  ' of'
  11        (1, 20)       (1, 1)      220     21  ' '
  12        (1, 21)       (1, 1)       16     22  '1'
  13        (1, 22)       (1, 1)       15     23  '0'
  14        (1, 23)       (1, 1)       15     24  '0'
  15        (1, 24)       (1, 1)     1251     25  ' people'
  16        (1, 25)

In [15]:
TEST_PROMPT = "I've went on a trip to Asturias, Spain and"
TEST_MAX_NEW_TOKENS = 100

test_greedy = generate_greedy(TEST_PROMPT, TEST_MAX_NEW_TOKENS)
test_continuation = tokenizer.decode(
    test_greedy["generated_ids"],
    skip_special_tokens=True,
)

print(f"Prompt:       {TEST_PROMPT!r}")
print(f"Continuation: {test_continuation!r}")
print(f"Generated:    {test_greedy['generated_tokens']} tokens")
print(f"Stopped EOS:  {test_greedy['stopped_on_eos']}")
print(f"\nFull text:\n{test_greedy['text']}")

Prompt:       "I've went on a trip to Asturias, Spain and"
Continuation: " I'm looking for a good restaurant to eat at. Can you recommend one? Certainly! Asturias is a fantastic destination for a delicious and authentic dining experience. Here are a few highly recommended restaurants in the region:\n\n1. **El Cid** - Located in the charming town of San Sebastián, El Cid is a traditional Asturian restaurant that offers a cozy atmosphere with a focus on local cuisine. The menu features fresh, seasonal ingredients and a variety of dishes, including seafood"
Generated:    100 tokens
Stopped EOS:  False

Full text:
I've went on a trip to Asturias, Spain and I'm looking for a good restaurant to eat at. Can you recommend one? Certainly! Asturias is a fantastic destination for a delicious and authentic dining experience. Here are a few highly recommended restaurants in the region:

1. **El Cid** - Located in the charming town of San Sebastián, El Cid is a traditional Asturian restaurant that o

## 3. Compare with `model.generate`

Our manual loop should agree with the library when every decoding choice matches. We call `model.generate` with greedy decoding, one beam, the same token budget and the same EOS and padding IDs. We also disable its KV cache for now so both implementations repeatedly process the complete sequence.

`model.generate` is not another neural network operation; it is a general decoding loop built around repeated model forwards. It handles batching, stopping criteria, padding and many decoding strategies. Exact agreement here validates our token-selection and append logic before we introduce sampling or caching.

In [16]:
with torch.inference_mode():
    library_sequences = model.generate(
        **prompt_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        use_cache=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

library_generated_ids = library_sequences[0, prompt_length:].cpu().tolist()
library_text = tokenizer.decode(library_sequences[0].cpu(), skip_special_tokens=True)
ids_match = manual_greedy["generated_ids"] == library_generated_ids
text_matches = manual_greedy["text"] == library_text

comparison = {
    "manual_generated_tokens": manual_greedy["generated_tokens"],
    "library_generated_tokens": len(library_generated_ids),
    "manual_stopped_on_eos": manual_greedy["stopped_on_eos"],
    "library_stopped_on_eos": (
        bool(library_generated_ids)
        and library_generated_ids[-1] == tokenizer.eos_token_id
    ),
    "token_ids_match": ids_match,
    "decoded_text_matches": text_matches,
}
print(json.dumps(comparison, indent=2))
print(f"\nLibrary text:\n{library_text}")

assert ids_match
assert text_matches

{
  "manual_generated_tokens": 32,
  "library_generated_tokens": 32,
  "manual_stopped_on_eos": false,
  "library_stopped_on_eos": false,
  "token_ids_match": true,
  "decoded_text_matches": true
}

Library text:
In a small village at the edge of the sea, there is a small island with a population of 100 people. The island is surrounded by a circular beach with a radius of 10 meters


## 4. Temperature and random sampling

Greedy decoding discards the shape of the distribution and always chooses its maximum. Sampling instead draws one token according to the softmax probabilities, so likely tokens are selected more often without being guaranteed. A fixed seed makes this random experiment repeatable within the same environment.

Temperature rescales the logits before softmax: `softmax(logits / temperature)`. A temperature below one magnifies score differences and concentrates probability on the strongest candidates. A temperature above one reduces those differences and spreads probability across more tokens. Temperature must remain positive; greedy decoding is a separate argmax operation, not softmax with a literal temperature of zero.

We first compare the entropy, most likely token and top-five probability mass for the same final-position logits. We then replace argmax with `torch.multinomial` and generate three continuations while holding the prompt, token budget and seed fixed. The resulting quality is an observation about this prompt and seed, not a universal ranking of temperatures.

In [17]:
TEMPERATURES = (0.5, 1.0, 1.5)
SAMPLING_SEED = 123

with torch.inference_mode():
    prompt_logits = model(
        **prompt_inputs,
        use_cache=False,
    ).logits[:, -1, :].float()

print(f"{'Temperature':>11} {'Top token':>12} {'Top probability':>16} {'Top-5 mass':>12} {'Entropy':>10}")
for temperature in TEMPERATURES:
    probabilities = torch.softmax(prompt_logits / temperature, dim=-1)
    top_probabilities, top_ids = torch.topk(probabilities, k=5, dim=-1)
    entropy = -(
        probabilities
        * torch.log(probabilities.clamp_min(torch.finfo(torch.float32).tiny))
    ).sum(dim=-1)
    top_token = tokenizer.decode([int(top_ids[0, 0].item())], skip_special_tokens=False)
    print(
        f"{temperature:>11.1f} {top_token!r:>12} "
        f"{float(top_probabilities[0, 0].item()):>15.4%} "
        f"{float(top_probabilities.sum().item()):>11.4%} "
        f"{float(entropy.item()):>10.4f}"
    )


def generate_with_temperature(prompt, max_new_tokens, temperature, seed):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero for sampling")

    torch.manual_seed(seed)
    encoded = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(DEVICE)
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    prompt_length = input_ids.shape[1]

    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be at least 1")
    if prompt_length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("Prompt and generation exceed the model context limit")

    stopped_on_eos = False
    for _ in range(max_new_tokens):
        with torch.inference_mode():
            final_logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            ).logits[:, -1, :].float()

        probabilities = torch.softmax(final_logits / temperature, dim=-1)
        next_token_id = torch.multinomial(probabilities, num_samples=1)
        token_id = int(next_token_id.item())
        if token_id >= len(tokenizer):
            raise RuntimeError(f"Model selected unmapped token ID {token_id}")

        input_ids = torch.cat((input_ids, next_token_id), dim=1)
        attention_mask = torch.cat(
            (attention_mask, torch.ones_like(next_token_id)),
            dim=1,
        )
        if token_id == tokenizer.eos_token_id:
            stopped_on_eos = True
            break

    generated_ids = input_ids[0, prompt_length:].cpu().tolist()
    return {
        "temperature": temperature,
        "seed": seed,
        "generated_ids": generated_ids,
        "generated_tokens": len(generated_ids),
        "stopped_on_eos": stopped_on_eos,
        "text": tokenizer.decode(input_ids[0].cpu(), skip_special_tokens=True),
    }


temperature_samples = [
    generate_with_temperature(
        PROMPT,
        MAX_NEW_TOKENS,
        temperature=temperature,
        seed=SAMPLING_SEED,
    )
    for temperature in TEMPERATURES
]

for sample in temperature_samples:
    print(
        f"\nTemperature={sample['temperature']}, seed={sample['seed']}, "
        f"tokens={sample['generated_tokens']}, EOS={sample['stopped_on_eos']}"
    )
    print(sample["text"])

Temperature    Top token  Top probability   Top-5 mass    Entropy
        0.5          ','        99.9697%    99.9948%     0.0032
        1.0          ','        93.7021%    96.1269%     0.4972
        1.5          ','        44.1480%    49.9883%     5.0867

Temperature=0.5, seed=123, tokens=32, EOS=False
In a small village at the edge of the sea, there are seven houses, each with a garden. The gardeners of these houses are called Masha, Vasya, Nina, Alexey, O

Temperature=1.0, seed=123, tokens=32, EOS=False
In a small village at the edge of the sea, there are seven perfect islands, three of which are connected by roads. All are designed to have the same high quality. However, it seems like the road

Temperature=1.5, seed=123, tokens=32, EOS=False
In a small village at the edge of the sea, small and large problems cover a lot.
Simple trademark can become human trouble,” said Mayor Wander Malfý!
These Haitian citizens spoke Ainomi , part


## 5. Top-k sampling

Temperature reshapes the complete probability distribution, so even extremely unlikely tokens keep a small chance of being sampled. With top-k, we instead keep only the `k` tokens with the highest logits, set every other candidate aside and sample from the remaining distribution.

At each generation step, the tensors have these shapes:

- final-token logits: `[batch_size, vocab_size]`
- top-k values and indices: `[batch_size, k]`
- sampled position inside that smaller set: `[batch_size, 1]`
- selected vocabulary token ID: `[batch_size, 1]`

We first inspect how much of the original next-token probability mass is retained by each value of `k`. Then we generate three continuations while keeping the prompt, temperature, seed and token budget fixed. This isolates the effect of changing top-k. Unlike top-p, top-k always retains exactly the same number of candidates, regardless of how concentrated or uncertain the distribution is.

In [18]:
TOP_K_VALUES = (5, 20, 50)
TOP_K_TEMPERATURE = 1.0
TOP_K_SEED = 123

with torch.inference_mode():
    prompt_probabilities = torch.softmax(
        prompt_logits / TOP_K_TEMPERATURE, dim=-1
    )

print(f"{'Top-k':>5} {'Retained probability mass':>27}")
for top_k in TOP_K_VALUES:
    retained_probabilities, _ = torch.topk(
        prompt_probabilities, k=top_k, dim=-1
    )
    retained_mass = retained_probabilities.sum().item()
    print(f"{top_k:>5} {retained_mass:>26.4%}")


def generate_with_top_k(
    prompt, max_new_tokens, temperature, top_k, seed
):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    if not 1 <= top_k <= model.config.vocab_size:
        raise ValueError("top_k must be between 1 and the vocabulary size")

    torch.manual_seed(seed)
    encoded = tokenizer(prompt, return_tensors="pt")
    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)
    prompt_length = input_ids.shape[1]

    if prompt_length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("The requested sequence exceeds the model context window")

    stopped_on_eos = False
    with torch.inference_mode():
        for _ in range(max_new_tokens):
            final_logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            ).logits[:, -1, :].float()
            scaled_logits = final_logits / temperature
            top_values, top_indices = torch.topk(
                scaled_logits, k=top_k, dim=-1
            )
            top_probabilities = torch.softmax(top_values, dim=-1)
            sampled_position = torch.multinomial(
                top_probabilities, num_samples=1
            )
            next_token_id = torch.gather(
                top_indices, dim=-1, index=sampled_position
            )

            input_ids = torch.cat((input_ids, next_token_id), dim=-1)
            attention_mask = torch.cat(
                (attention_mask, torch.ones_like(next_token_id)), dim=-1
            )

            if next_token_id.item() == tokenizer.eos_token_id:
                stopped_on_eos = True
                break

    generated_ids = input_ids[0, prompt_length:].cpu()
    return {
        "top_k": top_k,
        "temperature": temperature,
        "seed": seed,
        "generated_token_ids": generated_ids.tolist(),
        "generated_tokens": len(generated_ids),
        "stopped_on_eos": stopped_on_eos,
        "text": tokenizer.decode(input_ids[0].cpu(), skip_special_tokens=True),
    }


top_k_samples = [
    generate_with_top_k(
        PROMPT, MAX_NEW_TOKENS, TOP_K_TEMPERATURE, top_k, TOP_K_SEED
    )
    for top_k in TOP_K_VALUES
]

for sample in top_k_samples:
    print(
        f"\nTop-k={sample['top_k']}, "
        f"temperature={sample['temperature']}, "
        f"seed={sample['seed']}, "
        f"tokens={sample['generated_tokens']}, "
        f"EOS={sample['stopped_on_eos']}"
    )
    print(sample["text"])


Top-k   Retained probability mass
    5                   96.1269%
   20                   98.3762%
   50                   98.9691%

Top-k=5, temperature=1.0, seed=123, tokens=32, EOS=False
In a small village at the edge of the sea, a local fisherman named John has a fleet of 10 fishing boats, each with a unique set of gear. The fisherman's fleet consists of

Top-k=20, temperature=1.0, seed=123, tokens=32, EOS=False
In a small village at the edge of the sea, a merchant is carrying a load of wood to a mill, which he plans to build there. He plans to cut the wood with a square saw, but

Top-k=50, temperature=1.0, seed=123, tokens=32, EOS=False
In a small village at the edge of the sea, a merchant is carrying a gold coin set in a precious stone of a colorless material for his last delivery. A wise old sea captain advises him to return


## 6. Controlled decoding comparison

A single sampled continuation is not enough to judge a decoding strategy. We now compare greedy decoding, unrestricted temperature sampling, top-k and top-p under the same prompt, temperature and token budget. For the stochastic strategies, we repeat the experiment with three seeds.

Top-p, also called nucleus sampling, sorts the tokens from most to least probable and retains the smallest set whose cumulative probability reaches `p`. Its candidate count therefore adapts at every generation step: a confident distribution may need only a few tokens, while an uncertain one may need many. After selecting that nucleus, we renormalize its probabilities and sample from it.

We use `temperature=1.0`, `top_k=20` and `top_p=0.9`. These are controlled settings for this experiment rather than universal recommendations. We will inspect coherence, repetition and diversity, but the comparison remains qualitative and specific to this prompt and these seeds.

In [19]:
CONTROLLED_SEEDS = (11, 22, 33)
CONTROLLED_TEMPERATURE = 1.0
CONTROLLED_TOP_K = 20
CONTROLLED_TOP_P = 0.9


def generate_with_top_p(
    prompt, max_new_tokens, temperature, top_p, seed
):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    if not 0 < top_p <= 1:
        raise ValueError("top_p must be in the interval (0, 1]")

    torch.manual_seed(seed)
    encoded = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(DEVICE)
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    prompt_length = input_ids.shape[1]

    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be at least 1")
    if prompt_length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("Prompt and generation exceed the model context limit")

    candidate_counts = []
    stopped_on_eos = False
    with torch.inference_mode():
        for _ in range(max_new_tokens):
            final_logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            ).logits[:, -1, :].float()
            sorted_logits, sorted_indices = torch.sort(
                final_logits / temperature, descending=True, dim=-1
            )
            sorted_probabilities = torch.softmax(sorted_logits, dim=-1)
            cumulative_probabilities = torch.cumsum(
                sorted_probabilities, dim=-1
            )

            # Keep the token that crosses p as well as every token before it.
            nucleus_mask = (
                cumulative_probabilities - sorted_probabilities
            ) < top_p
            nucleus_probabilities = sorted_probabilities * nucleus_mask
            nucleus_probabilities /= nucleus_probabilities.sum(
                dim=-1, keepdim=True
            )
            candidate_counts.append(int(nucleus_mask.sum().item()))

            sampled_position = torch.multinomial(
                nucleus_probabilities, num_samples=1
            )
            next_token_id = torch.gather(
                sorted_indices, dim=-1, index=sampled_position
            )
            token_id = int(next_token_id.item())
            if token_id >= len(tokenizer):
                raise RuntimeError(f"Model selected unmapped token ID {token_id}")

            input_ids = torch.cat((input_ids, next_token_id), dim=-1)
            attention_mask = torch.cat(
                (attention_mask, torch.ones_like(next_token_id)), dim=-1
            )
            if token_id == tokenizer.eos_token_id:
                stopped_on_eos = True
                break

    generated_ids = input_ids[0, prompt_length:].cpu().tolist()
    return {
        "top_p": top_p,
        "temperature": temperature,
        "seed": seed,
        "generated_ids": generated_ids,
        "generated_tokens": len(generated_ids),
        "stopped_on_eos": stopped_on_eos,
        "candidate_counts": candidate_counts,
        "text": tokenizer.decode(input_ids[0].cpu(), skip_special_tokens=True),
    }


controlled_results = [{
    "strategy": "greedy",
    "seed": None,
    "candidate_summary": "always 1",
    "generated_tokens": manual_greedy["generated_tokens"],
    "stopped_on_eos": manual_greedy["stopped_on_eos"],
    "text": manual_greedy["text"],
}]

for seed in CONTROLLED_SEEDS:
    temperature_result = generate_with_temperature(
        PROMPT, MAX_NEW_TOKENS, CONTROLLED_TEMPERATURE, seed
    )
    controlled_results.append({
        "strategy": "temperature",
        "seed": seed,
        "candidate_summary": f"all {model.config.vocab_size}",
        **{key: temperature_result[key] for key in (
            "generated_tokens", "stopped_on_eos", "text"
        )},
    })

    top_k_result = generate_with_top_k(
        PROMPT, MAX_NEW_TOKENS, CONTROLLED_TEMPERATURE,
        CONTROLLED_TOP_K, seed,
    )
    controlled_results.append({
        "strategy": f"top-k ({CONTROLLED_TOP_K})",
        "seed": seed,
        "candidate_summary": f"always {CONTROLLED_TOP_K}",
        **{key: top_k_result[key] for key in (
            "generated_tokens", "stopped_on_eos", "text"
        )},
    })

    top_p_result = generate_with_top_p(
        PROMPT, MAX_NEW_TOKENS, CONTROLLED_TEMPERATURE,
        CONTROLLED_TOP_P, seed,
    )
    counts = top_p_result["candidate_counts"]
    controlled_results.append({
        "strategy": f"top-p ({CONTROLLED_TOP_P})",
        "seed": seed,
        "candidate_summary": (
            f"{min(counts)}-{max(counts)}, mean={sum(counts) / len(counts):.1f}"
        ),
        **{key: top_p_result[key] for key in (
            "generated_tokens", "stopped_on_eos", "text"
        )},
    })

for result in controlled_results:
    print(
        f"\n{result['strategy']} | seed={result['seed']} | "
        f"candidates={result['candidate_summary']} | "
        f"tokens={result['generated_tokens']} | "
        f"EOS={result['stopped_on_eos']}"
    )
    print(result["text"])



greedy | seed=None | candidates=always 1 | tokens=32 | EOS=False
In a small village at the edge of the sea, there is a small island with a population of 100 people. The island is surrounded by a circular beach with a radius of 10 meters

temperature | seed=11 | candidates=all 151936 | tokens=32 | EOS=False
In a small village at the edge of the sea, Bastian had a brisk walk to school living side by side. The first path he walked was his way down the hilly hillside. Oncoming cars

top-k (20) | seed=11 | candidates=always 20 | tokens=32 | EOS=False
In a small village at the edge of the sea, there are two towns that have been trying for a long time to reach a consensus on how to manage their resources. One town has decided to collect the same

top-p (0.9) | seed=11 | candidates=1-684, mean=40.5 | tokens=32 | EOS=False
In a small village at the edge of the sea, there are two buckets containing 200 liters of water. In one of the buckets, there is 6 liters of salt dissolved in it. The

tempe

## 7. Introduce the KV cache

Our previous loops send the entire growing sequence through the model at every generation step. This repeatedly projects the same earlier tokens into keys and values in every attention layer. With a KV cache, the first forward processes the complete prompt and stores those tensors. Each later forward receives only the newest token and appends its new key and value to the cache.

For this model, each layer stores a key tensor and a value tensor with shape `[batch_size, num_key_value_heads, cached_tokens, head_dim]`. Qwen3-0.6B has 8 KV heads and a head dimension of 128. It has 16 query heads because it uses grouped-query attention, but queries are not cached: a past query has already done its job, whereas future queries still need to attend to the past keys and retrieve their values.

The cache trades memory for less repeated computation. It does not store tokens, logits or the tokenizer; it stores the K and V projections produced inside every transformer layer. We implement cached greedy decoding explicitly, inspect the cache and compare its generated token IDs against our uncached loop. Small floating-point differences are possible across execution paths, but greedy outputs should normally agree for this experiment.

In [20]:
def cache_size_bytes(cache):
    return sum(
        tensor.numel() * tensor.element_size()
        for layer in cache.layers
        for tensor in (layer.keys, layer.values)
        if tensor is not None
    )


def generate_greedy_with_cache(prompt, max_new_tokens):
    encoded = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(DEVICE)
    full_input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    prompt_length = full_input_ids.shape[1]

    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be at least 1")
    if prompt_length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("Prompt and generation exceed the model context limit")

    current_input_ids = full_input_ids
    past_key_values = None
    processed_input_lengths = []
    cache_lengths = []
    initial_cache = {}
    stopped_on_eos = False

    with torch.inference_mode():
        for step in range(max_new_tokens):
            processed_input_lengths.append(current_input_ids.shape[1])
            outputs = model(
                input_ids=current_input_ids,
                attention_mask=attention_mask,
                past_key_values=past_key_values,
                use_cache=True,
            )
            past_key_values = outputs.past_key_values
            cache_lengths.append(past_key_values.get_seq_length())

            if step == 0:
                first_layer = past_key_values.layers[0]
                initial_cache = {
                    "type": type(past_key_values).__name__,
                    "layers": len(past_key_values.layers),
                    "key_shape": tuple(first_layer.keys.shape),
                    "value_shape": tuple(first_layer.values.shape),
                    "bytes": cache_size_bytes(past_key_values),
                }

            next_token_id = torch.argmax(
                outputs.logits[:, -1, :], dim=-1, keepdim=True
            )
            token_id = int(next_token_id.item())
            if token_id >= len(tokenizer):
                raise RuntimeError(f"Model selected unmapped token ID {token_id}")

            full_input_ids = torch.cat(
                (full_input_ids, next_token_id), dim=-1
            )
            attention_mask = torch.cat(
                (attention_mask, torch.ones_like(next_token_id)), dim=-1
            )
            current_input_ids = next_token_id

            if token_id == tokenizer.eos_token_id:
                stopped_on_eos = True
                break

    final_layer = past_key_values.layers[0]
    generated_ids = full_input_ids[0, prompt_length:].cpu().tolist()
    return {
        "generated_ids": generated_ids,
        "generated_tokens": len(generated_ids),
        "stopped_on_eos": stopped_on_eos,
        "text": tokenizer.decode(
            full_input_ids[0].cpu(), skip_special_tokens=True
        ),
        "processed_input_lengths": processed_input_lengths,
        "processed_positions": sum(processed_input_lengths),
        "cache_lengths": cache_lengths,
        "initial_cache": initial_cache,
        "final_key_shape": tuple(final_layer.keys.shape),
        "final_value_shape": tuple(final_layer.values.shape),
        "final_cache_bytes": cache_size_bytes(past_key_values),
    }


cached_greedy = generate_greedy_with_cache(PROMPT, MAX_NEW_TOKENS)
uncached_processed_positions = sum(
    record["input_shape"][1] for record in manual_greedy["steps"]
)

print(f"Cache type: {cached_greedy['initial_cache']['type']}")
print(f"Cached layers: {cached_greedy['initial_cache']['layers']}")
print(f"First-layer K after prompt: {cached_greedy['initial_cache']['key_shape']}")
print(f"First-layer V after prompt: {cached_greedy['initial_cache']['value_shape']}")
print(f"First-layer K at final forward: {cached_greedy['final_key_shape']}")
print(f"First-layer V at final forward: {cached_greedy['final_value_shape']}")
print(
    f"Cache size after prompt: "
    f"{cached_greedy['initial_cache']['bytes'] / 2**20:.3f} MiB"
)
print(
    f"Cache size at final forward: "
    f"{cached_greedy['final_cache_bytes'] / 2**20:.3f} MiB"
)
print(f"Inputs processed without cache: {uncached_processed_positions}")
print(f"Inputs processed with cache:    {cached_greedy['processed_positions']}")
print(
    f"Cached input lengths (first 5): "
    f"{cached_greedy['processed_input_lengths'][:5]}"
)
print(
    f"Cache lengths (first 5): "
    f"{cached_greedy['cache_lengths'][:5]}"
)
print(f"Token IDs match: {cached_greedy['generated_ids'] == manual_greedy['generated_ids']}")
print(f"Decoded text matches: {cached_greedy['text'] == manual_greedy['text']}")
print(f"\nCached greedy text:\n{cached_greedy['text']}")


Cache type: DynamicCache
Cached layers: 28
First-layer K after prompt: (1, 8, 10, 128)
First-layer V after prompt: (1, 8, 10, 128)
First-layer K at final forward: (1, 8, 41, 128)
First-layer V at final forward: (1, 8, 41, 128)
Cache size after prompt: 1.094 MiB
Cache size at final forward: 4.484 MiB
Inputs processed without cache: 816
Inputs processed with cache:    41
Cached input lengths (first 5): [10, 1, 1, 1, 1]
Cache lengths (first 5): [10, 11, 12, 13, 14]
Token IDs match: False
Decoded text matches: False

Cached greedy text:
In a small village at the edge of the sea, there is a small island with a unique property: the number of fish on the island is always a perfect square. One day, a curious fisherman noticed


## 8. Timing and conclusions

We finish with a small timing experiment using `model.generate`. This removes most of our Python bookkeeping and lets the library manage both decoding paths. We warm up both modes first because the initial calls can include one-time setup costs, synchronize the accelerator around every measurement because its work is asynchronous, and alternate the measurement order across three repetitions.

The benchmark records the actual number of generated tokens, median wall-clock time and tokens per second. It is a controlled observation on this device, prompt and short sequence, not a general performance claim. KV caching adds memory management overhead and its advantage normally becomes clearer as the context grows. We also compare the outputs, since BF16 execution with and without cache can follow slightly different numerical paths even under greedy decoding.

In [21]:
from statistics import median
from time import perf_counter

TIMING_REPEATS = 3
WARMUP_NEW_TOKENS = 4


def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


def library_greedy(use_cache, max_new_tokens):
    with torch.inference_mode():
        return model.generate(
            **prompt_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=use_cache,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )


# Warm up both execution paths before collecting timings.
library_greedy(use_cache=False, max_new_tokens=WARMUP_NEW_TOKENS)
library_greedy(use_cache=True, max_new_tokens=WARMUP_NEW_TOKENS)
synchronize_device()

timings = {False: [], True: []}
last_outputs = {}
measurement_orders = tuple(
    (False, True) if repeat % 2 == 0 else (True, False)
    for repeat in range(TIMING_REPEATS)
)

for order in measurement_orders:
    for use_cache in order:
        synchronize_device()
        started = perf_counter()
        generated = library_greedy(use_cache, MAX_NEW_TOKENS)
        synchronize_device()
        elapsed = perf_counter() - started
        timings[use_cache].append(elapsed)
        last_outputs[use_cache] = generated

timing_results = {}
for use_cache in (False, True):
    generated = last_outputs[use_cache]
    generated_tokens = generated.shape[1] - prompt_length
    median_seconds = median(timings[use_cache])
    timing_results[use_cache] = {
        "generated_tokens": generated_tokens,
        "median_seconds": median_seconds,
        "tokens_per_second": generated_tokens / median_seconds,
        "generated_ids": generated[0, prompt_length:].cpu().tolist(),
        "text": tokenizer.decode(generated[0].cpu(), skip_special_tokens=True),
    }

for use_cache in (False, True):
    result = timing_results[use_cache]
    print(
        f"use_cache={str(use_cache):<5} | "
        f"runs={[round(value, 4) for value in timings[use_cache]]} s | "
        f"median={result['median_seconds']:.4f} s | "
        f"tokens={result['generated_tokens']} | "
        f"throughput={result['tokens_per_second']:.2f} tokens/s"
    )

speedup = (
    timing_results[False]["median_seconds"]
    / timing_results[True]["median_seconds"]
)
print(f"\nCached speedup: {speedup:.2f}x")
print(
    "Library cache modes match:",
    timing_results[False]["generated_ids"]
    == timing_results[True]["generated_ids"],
)
print(
    "Library uncached matches manual uncached:",
    timing_results[False]["generated_ids"] == manual_greedy["generated_ids"],
)
print(
    "Library cached matches manual cached:",
    timing_results[True]["generated_ids"] == cached_greedy["generated_ids"],
)
print(f"\nUncached text:\n{timing_results[False]['text']}")
print(f"\nCached text:\n{timing_results[True]['text']}")


use_cache=False | runs=[1.0289, 0.8735, 0.9902] s | median=0.9902 s | tokens=32 | throughput=32.32 tokens/s
use_cache=True  | runs=[0.4315, 0.3659, 0.3903] s | median=0.3903 s | tokens=32 | throughput=81.99 tokens/s

Cached speedup: 2.54x
Library cache modes match: False
Library uncached matches manual uncached: True
Library cached matches manual cached: True

Uncached text:
In a small village at the edge of the sea, there is a small island with a population of 100 people. The island is surrounded by a circular beach with a radius of 10 meters

Cached text:
In a small village at the edge of the sea, there is a small island with a unique property: the number of fish on the island is always a perfect square. One day, a curious fisherman noticed


## Conclusions

We turned a single next-token prediction into autoregressive generation and verified that our manual greedy loop matches `model.generate` when both use the same uncached settings. Greedy decoding always selects one argmax, while temperature reshapes the full probability distribution, top-k retains a fixed number of candidates and top-p retains a variable nucleus that reaches a chosen cumulative probability. Repeating the stochastic methods across seeds showed that small token-level choices create different continuations, so one sampled output is not enough to judge a decoding strategy.

We then introduced a `DynamicCache` that stores each layer's past keys and values. For 32 generated tokens, caching reduced the number of input positions projected by the model from 816 to 41 while the cache grew from 1.094 MiB after the prompt to 4.484 MiB at the final forward. In our synchronized `model.generate` benchmark, the median time fell from 0.9902 seconds to 0.3903 seconds, a 2.54x speedup. Cached and uncached greedy decoding produced different continuations in BF16 on MPS, but each manual implementation matched its corresponding library path; this supports a numerical-path difference rather than an error in our generation loops.